# 04. Transfer Learning (Feature Extraction / Frozen Backbones)

Compares linear probing across pretrained backbones: VGG16, MobileNetV2, ResNet50, EfficientNet-B0, and ViT-B/16 with frozen feature extractors and trained classification heads.

In [ ]:
import torch
import torch.nn as nn
from cv_benchmark.utils import set_seed, get_device, count_parameters
from cv_benchmark.data import get_dataloaders
from cv_benchmark.models import create_model
from cv_benchmark.training import Trainer
from cv_benchmark.evaluation import evaluate_model

set_seed(42)
device = get_device("cuda")

models_to_evaluate = ["mobilenet_v2", "resnet50", "efficientnet_b0", "vgg16", "vit_b_16"]
results = {}

for m_name in models_to_evaluate:
    print(f"\n=== Running Feature Extraction for {m_name} ===")
    batch_size = 32 if "vit" in m_name else 64
    loaders = get_dataloaders(model_name=m_name, batch_size=batch_size, image_size=(224, 224))
    model, head_names = create_model(m_name, num_classes=100, pretrained=True, mode="feature_extraction")
    tot_p, tr_p = count_parameters(model)
    print(f"{m_name} -> Total: {tot_p:,} | Trainable: {tr_p:,}")

    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
    trainer = Trainer(
        model=model,
        train_loader=loaders["train"],
        val_loader=loaders["val"],
        criterion=nn.CrossEntropyLoss(),
        optimizer=optimizer,
        device=device,
        experiment_name=f"{m_name}_feature_extract",
    )
    trainer.fit(epochs=8)
    metrics = evaluate_model(model, loaders["test"], device=device, num_classes=100)
    results[m_name] = metrics
    print(f"{m_name} Test Top-1 Accuracy: {metrics['top1_accuracy']}%")